# Finetune Resnet50 for Quantization

In this notebook we will finetune a pre-trained Resnet50 with the CIFAR10 dataset to then use the result to explore Post Training Quantization.

## 🛠️ Supported Hardware

✅ AMD Instinct™ Accelerators
✅ AMD Radeon™ RX/PRO Graphics Cards
✅ AMD EPYC™ Processors
✅ AMD Ryzen™ (AI) Processors

Suggested hardware: **AI PC powered by AMD Ryzen™ AI Processors**. This notebook benefits from having a GPU.

## ⚡ Recommended Software Environment

::::{tab-set}

:::{tab-item} Linux
- [Install Docker container](https://amdresearch.github.io/aup-ai-tutorials//env/env-gpu.html)
- [Install PyTorch](https://amdresearch.github.io/aup-ai-tutorials//env/env-cpu.html)
:::

:::{tab-item} Windows
- [Install Direct-ML](https://amdresearch.github.io/aup-ai-tutorials//env/env-gpu-windows.html)
- [Install PyTorch](https://amdresearch.github.io/aup-ai-tutorials//env/env-cpu.html)
:::
::::

## 🎯 Goals

- Finetune Resnet50 for the CIFAR dataset
- Export finetuned model as ONNX format

:::{seealso}
- [The CIFAR-10 dataset](https://www.cs.toronto.edu/~kriz/cifar.html)
- [Deep Residual Learning for Image Recognition](https://arxiv.org/pdf/1512.03385)
:::

## Import Packages

Let us start by importing the Python packages we will use in this notebook.

In [ ]:
import random
import os
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.models import ResNet50_Weights, resnet50

## Prepare the Model

First of all, we get the device and use the GPU if available

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'Using {device=}')

### Load model for re-training using transfer learning

The pre-trained ResNet-50 model is trained on 1,000 class [ImageNet dataset](https://www.image-net.org/). By default it has fully connected (FC) layer of output size 1,000. This means that it produces a 1,000-dimensional vector, where each dimension corresponds to a class in the ImageNet dataset.

We use transfer learning to select a set of pre-trained weights for the model and then customize the model's classifier by replacing its FC layers. The modification includes adding two linear layers, one with 2,048 input features and 64 output features, followed by a ReLU activation function, and another linear layer with 64 input features and 10 output features. This adaptation transforms the ResNet-50 model into a classifier suitable for a specific task with 10 classes. 

In [ ]:
resnet = resnet50(weights=ResNet50_Weights.DEFAULT)

resnet.fc = torch.nn.Sequential(torch.nn.Linear(resnet.fc.in_features, 64), torch.nn.ReLU(inplace=True), torch.nn.Linear(64, 10))
resnet = resnet.to(device)
print(f"Modified Output Layer: {resnet.fc}")

Modified Output Layer: Sequential(
  (0): Linear(in_features=2048, out_features=64, bias=True)
  (1): ReLU(inplace=True)
  (2): Linear(in_features=64, out_features=10, bias=True)
)


## Model re-training

Let start by setting a known seed for reproducible results

In [4]:
random.seed(0)
torch.manual_seed(0)
torch.cuda.manual_seed(0)

Let us retrain the [ResNet-50 model](https://arxiv.org/pdf/1512.03385.pdf) from PyTorch Hub using the CIFAR-10 dataset.

The CIFAR-10 dataset is used to retrain the default model using the [transfer learning technique](https://www.youtube.com/watch?v=BqqfQnyjmgg&list=PLo2EIpI_JMQtNtKNFFSMNIZwspj8H7-sQ&index=3).

Create the train and validation dataloaders to train and validate the model.

In [ ]:
data_dir= os.path.join('data')
batch_size = 100

transform = transforms.Compose(
    [transforms.Pad(4), transforms.RandomHorizontalFlip(), transforms.RandomCrop(32), transforms.ToTensor()]
)

train_dataset = torchvision.datasets.CIFAR10(root=data_dir, train=True, transform=transform, download=True)
val_dataset = torchvision.datasets.CIFAR10(root=data_dir, train=False, transform=transforms.ToTensor())

train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
val_loader = torch.utils.data.DataLoader(dataset=val_dataset, batch_size=batch_size, shuffle=False)

Now, we define the hyper parameters for re-training. This is the learning rate, the loss function and the optimizer.

In [ ]:
learning_rate = 0.001
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(resnet.parameters(), lr=learning_rate)

The training process runs over 500 images with a `batch_size` of 100, i.e., over the total 50,000 images in the train set.

The training process takes approximately 10 minutes to complete each epoch. Number of epochs can be varied to optimize the accuracy of the model.

In [7]:
def update_lr(optimizer, lr):
    for param_group in optimizer.param_groups:
        param_group["lr"] = lr

def model_train(epochs):
    total_step = len(train_loader)
    curr_lr = learning_rate
    for epoch in range(epochs):
        correct = 0
        total = 0
        for i, (images, labels) in enumerate(train_loader):
            images = images.to(device)
            labels = labels.to(device)
            # Forward pass
            outputs = resnet(images)
            loss = criterion(outputs, labels)
            _, predicted = torch.max(outputs.data, 1)
            # Backward and optimize
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            accuracy = 100 * correct / total
            if (i + 1) % 250 == 0:
                print(f"Epoch [{epoch + 1}/{epochs}], Step [{i+1}/{total_step}] Accuracy {accuracy:.2f}% Loss: {loss.item():.4f}")
        # Decay learning rate
        if (epoch + 1) % 20 == 0:
            curr_lr /= 5
            update_lr(optimizer, curr_lr)

Run the training loop for 25 epochs.

In [ ]:
model_train(25)

Epoch [1/12], Step [250/500] Accuracy 60.54% Loss: 0.8284
Epoch [1/12], Step [500/500] Accuracy 67.73% Loss: 0.5694
Epoch [2/12], Step [250/500] Accuracy 78.76% Loss: 0.8322
Epoch [2/12], Step [500/500] Accuracy 79.35% Loss: 0.4815
Epoch [3/12], Step [250/500] Accuracy 81.68% Loss: 0.4979
Epoch [3/12], Step [500/500] Accuracy 82.16% Loss: 0.4648
Epoch [4/12], Step [250/500] Accuracy 83.56% Loss: 0.5411
Epoch [4/12], Step [500/500] Accuracy 83.75% Loss: 0.4640
Epoch [5/12], Step [250/500] Accuracy 85.24% Loss: 0.4269
Epoch [5/12], Step [500/500] Accuracy 85.00% Loss: 0.4453
Epoch [6/12], Step [250/500] Accuracy 86.28% Loss: 0.3284
Epoch [6/12], Step [500/500] Accuracy 85.92% Loss: 0.4442
Epoch [7/12], Step [250/500] Accuracy 86.65% Loss: 0.3735
Epoch [7/12], Step [500/500] Accuracy 86.67% Loss: 0.4445
Epoch [8/12], Step [250/500] Accuracy 87.35% Loss: 0.2556
Epoch [8/12], Step [500/500] Accuracy 87.32% Loss: 0.5832
Epoch [9/12], Step [250/500] Accuracy 88.26% Loss: 0.4089
Epoch [9/12], 

Once the model is trained, we will evaluate it with the validation dataset. These are images that the model has not seen before.

In [ ]:
def model_test():
    resnet.eval()
    with torch.no_grad():
        correct = 0
        total = 0
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = resnet(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        accuracy = 100 * correct / total
        print("Accuracy of the model on the test images: {} %".format(accuracy))

model_test()

Accuracy of the model on the test images: 86.83 %


Now, we can store the trained Pytorch model.

In [11]:
models_dir = os.path.join('onnx')
resnet.to("cpu")

if not os.path.isdir(models_dir):
    os.mkdir(models_dir)

model_path = os.path.join(models_dir, 'resnet_trained_for_cifar10.pt')
torch.save(resnet, model_path)

After completing the training process, the trained ResNet-50 model on the CIFAR-10 dataset is saved at the following location: `onnx/resnet_trained_for_cifar10.pt`.

## Export the Model to ONNX Format

Run the following cell to save the trained model as an ONNX model. We will use the ONNX model for quantization notebook.

In [12]:
dummy_inputs = torch.randn(1, 3, 32, 32)
input_names = ['input']
output_names = ['output']
dynamic_axes = {'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
onnx_model_path = f"{models_dir}/resnet_trained_for_cifar10.onnx"
torch.onnx.export(
    resnet,
    dummy_inputs,
    onnx_model_path,
    export_params=True,
    opset_version=17,
    input_names=input_names,
    output_names=output_names,
    dynamic_axes=dynamic_axes,
)

After completing this process, observe the trained ResNet-50 model on the CIFAR-10 dataset is saved at the following location in ONNX format: `onnx/resnet_trained_for_cifar10.onnx`.

## Conclusion

In this notebook, we explore how to modify a Resnet50 model to classify only 10 classes. We also applied transfer learning to finetune the model to classify for a different dataset. Finally, we saved the trained model as Pytorch and ONNX format.

----------
Copyright (C) 2025 Advanced Micro Devices, Inc. All rights reserved.

SPDX-License-Identifier: MIT